# Plotting Sensitivity

Here we show how to plot a sensitivity proxy as a function of Q. This is a measure of how strongly the reflectivity at each Q point depends on the parameters marked as variable, the actual values are arbritary outside of comparison of sensitivity within a given system.

In [ ]:
%matplotlib inline

import numpy as np

from refnx.reflect import SLD, Slab
from refnx.analysis import Parameter

from hogben.models.samples import Sample

# Reduce size of plots for the notebook.
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7,5)
plt.rcParams['figure.dpi'] = 100

## Construct a model

Here we show the sensitivity of a simple hydrogenous lipid bilayer in two water contrasts. By changing the refnx model, you can investigate the sensitivity of your own system.

In [ ]:
# Define SLD's of materials
D2O = SLD(6.36, name='D2O')
H2O = SLD(-0.56, name='H2O')
substrate = SLD(2.07, name='Silicon')


# Define the parameters of the system. Any parameter which is set to vary (i.e. vary=True) will be used to calculate the sensitivity
sio2_thickness = Parameter(20, 'SiO2 Thickness', vary=False, bounds=(0, 1000))
sio2_roughness = Parameter(2.5, 'SiO2 Roughness', vary=False, bounds=(0, 10))
sio2_sld = Parameter(3.47, 'SiO2 SLD', vary=False, bounds=(2, 5))
sio2_solvation = Parameter(0.2, 'SiO2 Solvation', vary=False, bounds=(0, 0.5))

lh_thick=Parameter(6.4,'Inner headgroup thickness', vary=True, bounds=(4, 14))
lt_thick=Parameter(14, 'Total tailgroup thickness', vary=True, bounds=(10, 30))

lh_sld = Parameter(2.0, 'SLD inner hg', bounds=(1.5, 2.5), vary=True)
lt_sld= Parameter(-0.25, 'SLD inner hg', bounds=(-0.3, -0.1), vary=True)

lipid_rough=Parameter(2.5,'Lipid Roughness', vary=False, bounds=(1, 4))

lipid_head_solvation=Parameter(0.3,'Lipid Head Solvation', vary=True, bounds=(0.1, 0.5))
lipid_tail_solvation=Parameter(0.01,'Lipid Tail Solvation', vary=True, bounds=(0.01, 0.1))

outer_roughness = Parameter(3, vary=False, name='Outer Roughness', bounds=(0, 10))


# Build the slabs of the model using the previously defined SLD's and parameters
sio2_slab = Slab(sio2_thickness, sio2_sld, sio2_roughness, vfsolv=sio2_solvation)
lipid_head = Slab(lh_thick,lh_sld,lipid_rough, vfsolv=lipid_head_solvation)
lipid_tail = Slab(lt_thick,lt_sld,lipid_rough, vfsolv=lipid_tail_solvation)


# Using the refnx syntax, create the structures
sample_d2o = substrate | sio2_slab | lipid_head | lipid_tail | lipid_tail | lipid_head | D2O(0,outer_roughness)
sample_h2o = substrate | sio2_slab | lipid_head | lipid_tail | lipid_tail | lipid_head | H2O(0,outer_roughness)


# Everything above here is pure refnx syntax. We now wrap the structures in the HOGBEN Sample class to calculate the sensitivity.
# Here we set up H2O, D2O and a combined sample with both contrasts.

sample_h = Sample([sample_h2o],bkg=[6e-6],dq=2)
sample_d = Sample([sample_d2o],bkg=[3e-6],dq=2)

sample_both = Sample([sample_h2o, sample_d2o],bkg=[6e-6, 3e-6],dq=2)


In [ ]:
#Define a number of Q points to look at the reflectivity and sensitivity
q=np.linspace(0.01, 0.3, 100)

In [ ]:
sample_d.plot_sensitivity_profile(q, min_sensitivity=1e-18);
# The min sensitivity can be used to make plots scale better when there are regions of near zero sensitivity.
# The default minimum sensitivity is 1e-20

In [ ]:
sample_h.plot_sensitivity_profile(q);

In [ ]:
sample_both.plot_sensitivity_profile(q);